<a href="https://colab.research.google.com/github/polmazon/tfm/blob/claude%2Ffestive-albattani-93veeb/scraper_competencia_honda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Motor de Web Scraping — Ofertas Financieras Competencia Automoción
**Honda Financial Services | TFM BSM Barcelona**

Extrae ofertas de financiación (TIN, TAE, comisión de apertura, cuota, plazo, etc.) de webs de competidores usando scraping + LLM (Claude).

## 1. Instalación de dependencias (ejecutar solo en Colab)

In [216]:
# Ejecuta esta celda la primera vez en Google Colab (tarda ~1 minuto)
!pip install -q requests beautifulsoup4 selenium pandas webdriver-manager openai
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt install -y -q ./google-chrome-stable_current_amd64.deb
print("Dependencias instaladas correctamente")

Reading package lists...
Building dependency tree...
Reading state information...
google-chrome-stable is already the newest version (149.0.7827.114-1).
0 upgraded, 0 newly installed, 0 to remove and 30 not upgraded.
Dependencias instaladas correctamente


## 2. Imports y configuración

In [217]:
import requests
import time
import json
import re
import pandas as pd
from datetime import date
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from openai import OpenAI

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


In [ ]:
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

CAMPOS_OFERTA = [
    "marca",
    "modelo",
    "precio_vehiculo",
    "cuota_mensual",
    "plazo_meses",
    "entrada",
    "tin",
    "tae",
    "comision_apertura",
    "valor_residual",
    "importe_financiado",
    "tipo_financiacion",
    "fecha_fin_oferta",
    "url",
    "fecha_extraccion"
]

print(f"API Key OpenAI cargada: {'OK' if OPENAI_API_KEY else 'ERROR — revisa los Secrets'}")

## 3. Funciones de scraping

In [218]:
def scrape_estatico(url, reintentos=3, pausa=2):
    for intento in range(reintentos):
        try:
            response = requests.get(url, headers=HEADERS, timeout=15)
            response.raise_for_status()
            return response.text
        except requests.RequestException as e:
            print(f"  [intento {intento+1}/{reintentos}] Error en {url}: {e}")
            time.sleep(pausa * (intento + 1))
    return None


def crear_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument(f"user-agent={HEADERS['User-Agent']}")
    driver_path = ChromeDriverManager().install()
    return webdriver.Chrome(service=Service(driver_path), options=options)


def scroll_hasta_el_final(driver, pausas=8):
    for i in range(pausas):
        driver.execute_script("window.scrollBy(0, document.body.scrollHeight);")
        time.sleep(1.5)
    driver.execute_script("window.scrollTo(0, 0);")


def scrape_dinamico(url, espera_extra=3, scroll=False):
    driver = crear_driver()
    try:
        driver.get(url)
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        time.sleep(espera_extra)
        if scroll:
            scroll_hasta_el_final(driver)
            time.sleep(2)
        return driver.page_source
    except Exception as e:
        print(f"  Error Selenium en {url}: {e}")
        return None
    finally:
        driver.quit()


def html_a_texto(html, seccion_especial=None, max_chars_legal=4000,
                 pagina_listado=False, umbral_tin=6000):
    """
    Extrae el fragmento relevante del HTML para enviarlo al LLM.

    pagina_listado=True: extrae desde el primer TIN hasta el FINAL del texto,
      sin límite de caracteres, para capturar todas las ofertas del listado.
    umbral_tin: solo aplica cuando pagina_listado=False. TINs más allá de esta
      posición se consideran carrusel y se ignoran.
    """
    if not html:
        return ""
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "nav", "header", "noscript"]):
        tag.decompose()
    texto = soup.get_text(separator=" ", strip=True)
    texto = re.sub(r'\s+', ' ', texto)

    cabecera = texto[:1500]

    # Sección especial (ej. Renault)
    if seccion_especial:
        idx = texto.find(seccion_especial)
        if idx != -1:
            print(f"  Sección '{seccion_especial[:40]}' encontrada en pos {idx}")
            return cabecera + " [...] " + texto[idx:idx + max_chars_legal]
        else:
            print(f"  AVISO: Sección especial no encontrada, usando fallback")

    # Buscar primer TIN
    pos = -1
    for kw in ["TIN:", "TIN :", "T.I.N"]:
        p = texto.find(kw)
        if p != -1:
            pos = p
            break

    if pos != -1:
        if pagina_listado:
            # Modo listado: sin límite, todo el texto desde el primer TIN hasta el final
            bloque = texto[max(0, pos - 200):]
            print(f"  [LISTADO] TIN en pos {pos} — extrayendo {len(bloque)} chars hasta el final")
            return cabecera + " [...] " + bloque
        elif pos < umbral_tin:
            inicio = max(0, pos - 500)
            bloque = texto[inicio:inicio + max_chars_legal]
            print(f"  Texto enviado al LLM: {len(cabecera) + len(bloque)} chars (TIN en pos {pos})")
            return cabecera + " [...] " + bloque
        else:
            print(f"  TIN encontrado en pos {pos} (carrusel, ignorado) — usando cabecera")

    print(f"  Texto enviado al LLM: {len(cabecera)} chars (sin texto legal propio)")
    return cabecera


print("Funciones de scraping definidas")

Funciones de scraping definidas


## 4. Extracción con LLM (Claude)

In [219]:
client = OpenAI(api_key=OPENAI_API_KEY)

PROMPT_SISTEMA = """Eres un experto en análisis de ofertas de financiación de automóviles en España.
Tu tarea es extraer información estructurada de textos de páginas web de concesionarios.
Devuelve SIEMPRE un JSON válido con los campos indicados.
Si un campo no aparece en el texto, devuelve null para ese campo.
No inventes datos. Solo extrae lo que esté explícitamente en el texto."""

PROMPT_CAMPOS = """
Para cada oferta devuelve un objeto JSON con estos campos:
- modelo: nombre comercial completo del modelo (incluyendo versión y kW si aparecen)
- tipo_combustible: "gasolina", "diésel", "híbrido", "híbrido enchufable", "eléctrico" o null
- precio_vehiculo: PVP al contado en € (número). Busca "PVP al contado", "precio al contado" o "PVP recomendado"
- cuota_mensual: cuota mensual en € (número)
- plazo_meses: duración en meses (número)
- entrada: entrada inicial en € (número, 0 si no hay)
- tin: TIN en % (número)
- tae: TAE en % (número)
- comision_apertura: importe de la comisión de apertura en € (número, 0 si es gratuita)
- porcentaje_comision_apertura: comisión de apertura en % sobre el capital (número o null)
- valor_residual: última cuota o valor residual en € (número o null)
- importe_financiado: capital total financiado en € (número)
- tipo_financiacion: nombre exacto del producto financiero
- fecha_fin_oferta: fecha límite en YYYY-MM-DD (string o null)

Devuelve SOLO este JSON:
{"ofertas": [ {...} ]}
"""


def _llamar_llm(trozo, marca, url, instruccion, max_tokens):
    prompt = f"""{instruccion}

Analiza el siguiente texto de la web de {marca} ({url}).
{PROMPT_CAMPOS}
TEXTO:
{trozo}"""
    respuesta = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=max_tokens,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": PROMPT_SISTEMA},
            {"role": "user", "content": prompt}
        ]
    )
    datos = json.loads(respuesta.choices[0].message.content)
    return [o for o in datos.get("ofertas", []) if o is not None]


def _anotar(ofertas, marca, url):
    for o in ofertas:
        o["marca"] = marca
        o["url"] = url
        o["fecha_extraccion"] = str(date.today())
    return ofertas


def _fusionar_sin_duplicados(listas):
    """Fusiona listas deduplicando por nombre de modelo normalizado completo."""
    vistos, resultado = set(), []
    for o in (o for lista in listas for o in lista):
        key = re.sub(r'\s+', ' ', (o.get("modelo") or "").strip().lower())
        if key and key not in vistos:
            vistos.add(key)
            resultado.append(o)
    return resultado


def extraer_oferta_con_llm(texto, marca, url, filtro_producto=None, pagina_listado=False):
    slug = url.rstrip("/").split("/")[-1]
    pista_modelo = (slug.replace("-easy-plus", "").replace("-easy-renting", "")
                    .replace("-easy", "").replace("-", " ").title())

    try:
        if filtro_producto:
            instruccion = (
                f"IMPORTANTE: Extrae ÚNICAMENTE la oferta principal del producto '{filtro_producto}' "
                f"para el modelo '{pista_modelo}'. Devuelve exactamente UNA oferta. "
                f"Ignora cualquier otro modelo o producto mencionado en el texto."
            )
            return _anotar(_llamar_llm(texto, marca, url, instruccion, 1500), marca, url)

        if pagina_listado:
            instruccion = (
                "Extrae TODAS las ofertas de financiación que encuentres en el texto legal. "
                "Cada bloque de condiciones legales corresponde a un modelo distinto. "
                "Devuelve una entrada por cada bloque/modelo con TIN o TAE propio."
            )
            # 3 trozos solapados para cubrir páginas con 20+ modelos
            n = len(texto)
            t1, t2 = n // 3, 2 * n // 3
            solape = 3000
            trozos = [
                texto[:t1 + solape],
                texto[t1 - solape:t2 + solape],
                texto[t2 - solape:]
            ]
            print(f"  [LISTADO] 3 llamadas: {len(trozos[0])} + {len(trozos[1])} + {len(trozos[2])} chars")
            resultados = []
            for i, trozo in enumerate(trozos):
                ofertas_i = _llamar_llm(trozo, marca, url, instruccion, 5000)
                print(f"  Llamada {i+1}/3: {len(ofertas_i)} ofertas")
                resultados.append(_anotar(ofertas_i, marca, url))
                if i < 2:
                    time.sleep(1)
            return _fusionar_sin_duplicados(resultados)

        instruccion = "Extrae la oferta de financiación principal que encuentres."
        return _anotar(_llamar_llm(texto, marca, url, instruccion, 1500), marca, url)

    except Exception as e:
        print(f"  Error LLM para {url}: {e}")
        return []


print("Cliente OpenAI (gpt-4o-mini) configurado")

Cliente OpenAI (gpt-4o-mini) configurado


## 5. Pipeline completo: scraping + extracción LLM

In [220]:
def procesar_url(url, marca, usar_selenium=False, scroll=False,
                 seccion_especial=None, filtro_producto=None,
                 pagina_listado=False, max_chars_legal=4000):
    print(f"Procesando: {marca} — {url}")
    html = scrape_dinamico(url, scroll=scroll) if usar_selenium else scrape_estatico(url)
    if not html:
        print(f"  No se pudo descargar {url}")
        return []
    texto = html_a_texto(
        html,
        seccion_especial=seccion_especial,
        max_chars_legal=max_chars_legal,
        pagina_listado=pagina_listado
    )
    ofertas = extraer_oferta_con_llm(
        texto, marca, url,
        filtro_producto=filtro_producto,
        pagina_listado=pagina_listado
    )
    print(f"  Ofertas encontradas: {len(ofertas)}")
    return ofertas


def descubrir_urls_toyota():
    """Extrae desde toyota.es/promociones todas las URLs con 'easy' sin 'renting'."""
    BASE = "https://www.toyota.es"
    print(f"Descubriendo URLs Toyota Easy desde {BASE}/promociones ...")
    html = scrape_estatico(f"{BASE}/promociones")
    if not html:
        print("  No se pudo descargar la página de promociones de Toyota")
        return []
    soup = BeautifulSoup(html, "html.parser")
    urls = []
    EXCLUIR = ["/promociones/toyota-easy-plus", "/promociones/toyota-easy",
               "/promociones/toyota-easy-complet"]
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if ("/promociones/" in href
                and "easy" in href
                and "renting" not in href
                and not any(href.endswith(ex.split("/")[-1]) and "/finance-insurance/" not in href
                            for ex in EXCLUIR)
                and "/finance-insurance/" not in href):
            url_completa = href if href.startswith("http") else BASE + href
            if url_completa not in urls:
                urls.append(url_completa)
    print(f"  URLs Easy encontradas: {len(urls)}")
    for u in urls:
        print(f"    {u}")
    return urls


print("Pipeline y auto-descubrimiento Toyota definidos")

Pipeline y auto-descubrimiento Toyota definidos


## 6. URLs de la competencia

In [221]:
COMPETENCIA = {
    "TOYOTA": {
        "selenium": False, "scroll": False,
        "seccion_especial": None, "filtro_producto": "Easy Plus",
        "pagina_listado": False, "max_chars_legal": 4000,
        "urls": []  # se auto-descubren desde toyota.es/promociones
    },
    "VOLKSWAGEN": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": None,
        "pagina_listado": True, "max_chars_legal": 60000,
        "urls": ["https://www.volkswagen.es/es/ofertas.html"]
    },
    "PEUGEOT": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": None,
        "pagina_listado": True, "max_chars_legal": 60000,
        "urls": ["https://www.peugeot.es/comprar/ofertas-del-momento.html"]
    },
    "RENAULT": {
        "selenium": False, "scroll": False,
        "seccion_especial": "CONDICIONES LEGALES PARA PENÍNSULA Y BALEARES",
        "filtro_producto": None,
        "pagina_listado": False, "max_chars_legal": 4000,
        "urls": [
            "https://promociones.renault.es/particulares/clio/",
            "https://promociones.renault.es/particulares/captur/",
            "https://promociones.renault.es/particulares/symbioz/",
            "https://promociones.renault.es/particulares/symbioz-glp/",
            "https://promociones.renault.es/particulares/austral/",
            "https://promociones.renault.es/particulares/arkana/",
            "https://promociones.renault.es/particulares/espace/",
            "https://promociones.renault.es/particulares/rafale/",
            "https://promociones.renault.es/particulares/rafale-phev/"
        ]
    },
    "NISSAN": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": None,
        "pagina_listado": True, "max_chars_legal": 60000,
        "urls": ["https://www.nissan.es/vehiculos/ofertas.html"]
    },
    "HYUNDAI": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": None,
        "pagina_listado": False, "max_chars_legal": 4000,
        "urls": [
            "https://www.hyundai.com/es/es/modelos/kona.html",
            "https://www.hyundai.com/es/es/modelos/tucson.html",
            "https://www.hyundai.com/es/es/modelos/nuevo-bayon.html",
            "https://www.hyundai.com/es/es/modelos/nuevo-i10.html",
            "https://www.hyundai.com/es/es/modelos/i20.html",
            "https://www.hyundai.com/es/es/modelos/i30.html",
            "https://www.hyundai.com/es/es/modelos/i30-fastback.html",
            "https://www.hyundai.com/es/es/modelos/i30wagon.html",
            "https://www.hyundai.com/es/es/modelos/nuevo-santafe-hev.html",
            "https://www.hyundai.com/es/es/modelos/kona-hibrido.html",
            "https://www.hyundai.com/es/es/modelos/tucson-hibrido.html",
            "https://www.hyundai.com/es/es/modelos/nuevo-santafe-phev.html",
            "https://www.hyundai.com/es/es/modelos/nuevo-tucson-phev.html",
            "https://www.hyundai.com/es/es/modelos/inster.html",
            "https://www.hyundai.com/es/es/modelos/kona-electrico.html",
            "https://www.hyundai.com/es/es/modelos/ioniq5.html",
            "https://www.hyundai.com/es/es/modelos/ioniq6.html",
            "https://www.hyundai.com/es/es/modelos/ioniq9.html"
        ]
    },
    "AUDI": {
        "selenium": True, "scroll": True,
        "seccion_especial": None, "filtro_producto": None,
        "pagina_listado": True, "max_chars_legal": 60000,
        "urls": ["https://www.audi.es/es/compra/promociones/"]
    }
}

print(f"Configuradas {len(COMPETENCIA)} marcas")

Configuradas 7 marcas


## 7. Ejecución del scraping

In [222]:
# Para probar solo una marca: MARCAS_A_EJECUTAR = ["VOLKSWAGEN"]
MARCAS_A_EJECUTAR = ["TOYOTA", "VOLKSWAGEN"]

todas_las_ofertas = []

for marca in MARCAS_A_EJECUTAR:
    config = COMPETENCIA[marca]
    print(f"\n{'='*50}\nMARCA: {marca}\n{'='*50}")

    # Toyota: auto-descubrimiento desde toyota.es/promociones
    if marca == "TOYOTA":
        urls = descubrir_urls_toyota()
        if not urls:
            print("  Fallback a URLs hardcodeadas")
            urls = config["urls"]
    else:
        urls = config["urls"]

    if not urls:
        print(f"  Sin URLs para {marca}, saltando.")
        continue

    for url in urls:
        ofertas = procesar_url(
            url, marca,
            usar_selenium=config["selenium"],
            scroll=config.get("scroll", False),
            seccion_especial=config.get("seccion_especial"),
            filtro_producto=config.get("filtro_producto"),
            pagina_listado=config.get("pagina_listado", False),
            max_chars_legal=config.get("max_chars_legal", 4000)
        )
        todas_las_ofertas.extend(ofertas)
        time.sleep(2)

print(f"\n{'='*50}")
print(f"RESUMEN: {len(todas_las_ofertas)} ofertas brutas extraídas")
marcas_con_datos = set(o["marca"] for o in todas_las_ofertas)
print(f"Marcas CON datos: {marcas_con_datos}")
marcas_sin_datos = set(MARCAS_A_EJECUTAR) - marcas_con_datos
if marcas_sin_datos:
    print(f"Marcas SIN datos: {marcas_sin_datos}")


MARCA: TOYOTA
Descubriendo URLs Toyota Easy desde https://www.toyota.es/promociones ...
  URLs Easy encontradas: 18
    https://www.toyota.es/promociones/toyota-c-hr-plus-easy-plus
    https://www.toyota.es/promociones/toyota-c-hr-140h-advance-easy-plus
    https://www.toyota.es/promociones/toyota-c-hr-plug-in-hybrid-220ph-advance-easy-plus
    https://www.toyota.es/promociones/yaris-ng-active-tech-easy-plus
    https://www.toyota.es/promociones/yaris-cross-hybrid-style-easy-plus
    https://www.toyota.es/promociones/corolla-hybrid-140h-active-plus-easy-plus
    https://www.toyota.es/promociones/corolla-sedan-hybrid-140h-style-plus-easy-plus
    https://www.toyota.es/promociones/corolla-touring-sports-hybrid-140h-easy-plus
    https://www.toyota.es/promociones/corolla-cross-hybrid-style-easy-plus
    https://www.toyota.es/promociones/toyota-bz4x-electric-4x2-advance-easy-plus
    https://www.toyota.es/promociones/aygo-x-cross-play-easy
    https://www.toyota.es/promociones/rav4-hybrid

In [223]:
df_bruto = pd.DataFrame(todas_las_ofertas)

if df_bruto.empty:
    print("No se han extraído ofertas.")
else:
    df = (
        df_bruto
        .drop_duplicates(subset=["marca", "modelo"], keep="first")
        .reset_index(drop=True)
    )

    CAMPOS_ORDENADOS = [
        "marca", "modelo", "tipo_combustible", "precio_vehiculo",
        "cuota_mensual", "plazo_meses", "entrada", "tin", "tae",
        "comision_apertura", "porcentaje_comision_apertura",
        "valor_residual", "importe_financiado", "tipo_financiacion",
        "fecha_fin_oferta", "url", "fecha_extraccion"
    ]
    cols = [c for c in CAMPOS_ORDENADOS if c in df.columns]
    df = df[cols]

    pd.set_option("display.max_columns", None)
    pd.set_option("display.max_rows", 100)

    print(f"Ofertas brutas: {len(df_bruto)} → tras deduplicar: {len(df)}")
    print(f"\nModelos por marca:")
    print(df.groupby("marca")["modelo"].count().to_string())
    display(df)

Ofertas brutas: 39 → tras deduplicar: 39

Modelos por marca:
marca
TOYOTA        18
VOLKSWAGEN    21


,marca,modelo,tipo_combustible,precio_vehiculo,cuota_mensual,plazo_meses,entrada,tin,tae,comision_apertura,porcentaje_comision_apertura,valor_residual,importe_financiado,tipo_financiacion,fecha_fin_oferta,url,fecha_extraccion
0,TOYOTA,Toyota C-HR+ Electric 250 Advance 77kWh,eléctrico,36075.00,250.0,48.0,11677.65,7.50,8.79,663.70,2.99,16991.74,22861.05,Toyota Easy Plus,2026-06-30,https://www.toyota.es/promociones/toyota-c-hr-...,2026-06-14
1,TOYOTA,Toyota C-HR 140H Advance,híbrido,32725.00,160.0,48.0,11309.80,7.50,8.71,0.00,2.99,17526.03,19415.20,Toyota Easy Plus,2026-06-30,https://www.toyota.es/promociones/toyota-c-hr-...,2026-06-14
2,TOYOTA,Toyota C-HR Plug-in Hybrid 220Ph Advance,híbrido enchufable,36075.00,200.0,48.0,11272.90,7.50,8.79,663.70,2.99,18966.53,22861.05,Toyota Easy Plus,2026-06-30,https://www.toyota.es/promociones/toyota-c-hr-...,2026-06-14
3,TOYOTA,Yaris Ng Active Tech,None,NaN,NaN,NaN,NaN,NaN,NaN,0.00,NaN,NaN,NaN,Toyota Easy Plus,2026-06-30,https://www.toyota.es/promociones/yaris-ng-act...,2026-06-14
4,TOYOTA,Yaris Cross Hybrid Style,híbrido,36075.00,140.0,48.0,10622.40,0.00,0.00,0.00,NaN,14542.81,NaN,Toyota Easy Plus,2026-06-30,https://www.toyota.es/promociones/yaris-cross-...,2026-06-14
5,TOYOTA,Corolla Hybrid 140H Active Plus,híbrido,36075.00,150.0,48.0,9829.40,7.50,8.79,0.00,2.99,13587.36,22861.05,Toyota Easy Plus,2026-06-30,https://www.toyota.es/promociones/corolla-hybr...,2026-06-14
6,TOYOTA,Corolla Sedan Hybrid 140H Style Plus,híbrido,36075.00,235.0,48.0,9695.95,7.50,8.79,663.70,2.99,14733.60,22861.05,Toyota Easy Plus,2026-06-30,https://www.toyota.es/promociones/corolla-seda...,2026-06-14
7,TOYOTA,Corolla Touring Sports Hybrid 140H,híbrido,36075.00,170.0,48.0,10210.45,7.50,8.79,0.00,2.99,15776.03,NaN,Toyota Easy Plus,2026-06-30,https://www.toyota.es/promociones/corolla-tour...,2026-06-14
8,TOYOTA,Corolla Cross Hybrid 200 Style,híbrido,36075.00,225.0,48.0,13957.80,7.50,8.79,663.70,2.99,19750.41,22861.05,Toyota Easy Plus,2026-06-30,https://www.toyota.es/promociones/corolla-cros...,2026-06-14
9,TOYOTA,Toyota Bz4X Electric 4X2 Advance,eléctrico,36075.00,300.0,48.0,15270.15,7.50,8.79,663.70,2.99,15174.38,22861.05,Toyota Easy Plus,2026-06-30,https://www.toyota.es/promociones/toyota-bz4x-...,2026-06-14


In [224]:
# Exportar a CSV
nombre_archivo = f"ofertas_competencia_{date.today().strftime('%Y%m%d')}.csv"
df.to_csv(nombre_archivo, index=False, encoding="utf-8-sig")
print(f"Guardado en: {nombre_archivo}")

# En Colab, descargar el archivo:
# from google.colab import files
# files.download(nombre_archivo)

Guardado en: ofertas_competencia_20260614.csv


In [225]:
# Resumen comparativo por marca
if not df.empty and "marca" in df.columns:
    resumen = df.groupby("marca").agg(
        num_ofertas=("modelo", "count"),
        tin_medio=("tin", "mean"),
        tae_medio=("tae", "mean"),
        cuota_min=("cuota_mensual", "min"),
        cuota_max=("cuota_mensual", "max")
    ).round(2)
    print(resumen)

            num_ofertas  tin_medio  tae_medio  cuota_min  cuota_max
marca                                                              
TOYOTA               18       7.03       8.24       99.0      495.0
VOLKSWAGEN           21       6.95       8.71      100.0      350.0
